
# Titanic: Woman-Child-Group (WCG) + XGBoost

## Introduction
The goal of this notebook is to implement a high-scoring strategy that typically achieves >0.82 on the Leaderboard, significantly outperforming standard model ensembles which often plateau around 0.78-0.80.

## comprehensive History of Attempts
This repository contains a long history of attempts to solve the Titanic challenge, spanning both R and Python implementations.

### Phase 1: The R Era (13 Iterations)
In the `2025-R-Attempts` directory, we conducted an exhaustive search for the best model.
*   **Champion Model (V4)**: A weighted soft voting ensemble (XGBoost + Random Forest + GLMnet) achieved the highest score of **0.78947**.
*   **Key Findings**:
    *   **Simplicity Wins**: Complex methods like Deep Learning (V6, 0.775) and Stacking (V9, 0.772) consistently underperformed the simpler V4 ensemble.
    *   **Variance is the Enemy**: Seed averaging (V11) stabilized the score (0.787) but smoothed out the peak performance of the lucky V4 seed.
    *   **Failed Experiments**: Pseudo-Labeling (V10) and Surgical Rules (V13) failed to generalize.

### Phase 2: The Python Era
We migrated the project to Python to leverage modern libraries and reproducibility.
*   **Benchmarking**: We tested Logistic Regression, Random Forest, SVM, XGBoost, and LightGBM.
*   **Feature Engineering**: We successfully replicated the "V4" features (Title, Deck, Family Size, Target-Encoded Survival Rates) in `titanic_utils.py`.
*   **Ensembling**: 
    *   Soft Voting (LGBM + XGB + SVM + RF) achieved **0.78229**.
    *   Optimized Seed Averaging (Bagging) also converged to **0.78229**.
    
### The Ceiling
Across both R and Python, and across widely different architectures (GLM, SVM, RF, GBM, Deep Learning), we have hit a hard ceiling around **0.78 - 0.79**. This suggests that standard "passenger-level" independent prediction has reached its limit given the noise in the data.

## The Logical Next Step: WCG + XGBoost
To break this ceiling, we must move beyond independent prediction and exploit the **strong correlation of fate within groups**.
This approach is based on the famous "Titanic WCG+XGBoost" kernel by Chris Deotte.

**Methodology:**
1.  **XGBoost Model**: Train a strong gradient boosting model on standard features to get a baseline prediction.
2.  **Woman-Child-Group (WCG) Post-Processing**: 
    *   Identify "groups" (Families or Ticket-holders) in the training set.
    *   **The Heuristic**: If a group of Women/Children in the training set **all died**, we assume that specific group was "doomed" (e.g., trapped in a cabin) and predict any Women/Children from that same group in the Test set will also die.
    *   Conversely, if they **all lived**, we predict survival for the Test set members.
    *   This "overrides" the model prediction, capturing specific "micro-fates" that a general model cannot learn as a rule.

This approach is the logical extension of our previous attempts because it directly addresses the "unexplained variance" that caused our ensembles to plateau.


In [ ]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import re
import warnings

warnings.filterwarnings('ignore')


In [ ]:

def load_data():
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")
    
    # Combine for processing
    train['is_train'] = 1
    test['is_train'] = 0
    test['Survived'] = np.nan
    full = pd.concat([train, test], sort=False).reset_index(drop=True)
    return full

full_df = load_data()
print(f"Data Loaded. Shape: {full_df.shape}")


## Feature Engineering

In [ ]:

def get_title(name):
    title_search = re.search(' ([A-Za-z]+)\.', name)
    if title_search:
        return title_search.group(1)
    return ""

def feature_engineering(df):
    # 1. Title
    df['Title'] = df['Name'].apply(get_title)
    # Normalize Titles
    df['Title'] = df['Title'].replace(['Mlle','Ms'], 'Miss') 
    df['Title'] = df['Title'].replace('Mme', 'Mrs') 
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 
                                       'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    
    # 2. Family Size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    
    # 3. Deck
    df['Deck'] = df['Cabin'].apply(lambda x: x[0] if pd.notna(x) else 'M') # M for Missing
    
    # 4. Surname (for WCG)
    df['Surname'] = df['Name'].apply(lambda x: x.split(',')[0])
    
    return df

full_df = feature_engineering(full_df)
print(full_df[['Name', 'Title', 'Surname', 'FamilySize', 'Deck']].head())


## Prepare Data for XGBoost

In [ ]:

def prepare_xgb_data(df):
    # Select features for XGBoost
    # We need to encode categorical variables
    df_enc = df.copy()
    
    # Label Encoding
    for col in ['Sex', 'Embarked', 'Title', 'Deck', 'Surname']:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        
    # Drop non-numeric or unneeded columns for model
    drop_cols = ['Name', 'Ticket', 'Cabin', 'PassengerId', 'is_train', 'Survived']
    X = df_enc.drop(drop_cols, axis=1)
    
    # Fill NA
    X = X.fillna(-999)
    
    return X, df_enc['Survived']

X_full, _ = prepare_xgb_data(full_df)

# Split back to train/test
train_mask = full_df['is_train'] == 1
X_train = X_full[train_mask]
y_train = full_df.loc[train_mask, 'Survived']
X_test = X_full[~train_mask]

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")


## Train XGBoost Model

In [ ]:

print("Training XGBoost...")
model = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
xgb_preds = model.predict(X_test)
print("Training Complete.")



## WCG Post-Processing
Here we apply the rule-based overrides.

**Rules:**
1. Identify "Woman-Child" (WC) candidates: Females and Masters (Boys).
2. Group by **Ticket** (stronger link) and **Surname** (weaker link).
3. If all WC in a Train group Died -> Predict Die for Test WC in that group.
4. If all WC in a Train group Lived -> Predict Live for Test WC in that group.


In [ ]:

def get_wcg_predictions(df, xgb_preds):
    df['Prediction'] = xgb_preds
    
    # Identify Woman and Child (Boys)
    df['IsWomanOrBoy'] = ((df['Title'] == 'Master') | (df['Sex'] == 'female'))
    
    # --- WCG Logic based on Surname ---
    train_df = df[df['is_train'] == 1]
    test_df = df[df['is_train'] == 0]
    
    # 1. Surname Logic
    surname_stats = train_df[train_df['IsWomanOrBoy']].groupby('Surname')['Survived'].agg(['count', 'mean', 'sum'])
    dead_surnames = surname_stats[(surname_stats['mean'] == 0.0) & (surname_stats['count'] > 0)].index.tolist()
    living_surnames = surname_stats[(surname_stats['mean'] == 1.0) & (surname_stats['count'] > 0)].index.tolist()
    
    # 2. Ticket Logic
    ticket_stats = train_df[train_df['IsWomanOrBoy']].groupby('Ticket')['Survived'].agg(['count', 'mean', 'sum'])
    dead_tickets = ticket_stats[(ticket_stats['mean'] == 0.0) & (ticket_stats['count'] > 0)].index.tolist()
    living_tickets = ticket_stats[(ticket_stats['mean'] == 1.0) & (ticket_stats['count'] > 0)].index.tolist()
    
    print(f"Found {len(dead_surnames)} dead surnames and {len(living_surnames)} living surnames.")
    print(f"Found {len(dead_tickets)} dead tickets and {len(living_tickets)} living tickets.")
    
    # --- Apply Overrides ---
    final_preds = df.loc[df['is_train'] == 0, 'Prediction'].copy()
    changes = 0
    
    for idx in test_df.index:
        row = df.loc[idx]
        if not row['IsWomanOrBoy']:
            continue
            
        original_pred = row['Prediction']
        new_pred = original_pred
        
        # Check Ticket
        if row['Ticket'] in dead_tickets:
            new_pred = 0
        elif row['Ticket'] in living_tickets:
            new_pred = 1
        else:
            # Check Surname if Ticket didn't decide
            if row['Surname'] in dead_surnames:
                new_pred = 0
            elif row['Surname'] in living_surnames:
                new_pred = 1
        
        if new_pred != original_pred:
            # print(f"Override: {row['Name']} -> {new_pred}")
            final_preds.loc[idx] = new_pred
            changes += 1
            
    print(f"WCG Logic modified {changes} predictions.")
    return final_preds

# Pass the FULL dataframe (with metadata) and the raw predictions
full_df_w_preds = full_df.copy()
full_df_w_preds.loc[~train_mask, 'Prediction'] = xgb_preds

final_preds = get_wcg_predictions(full_df_w_preds, full_df_w_preds.loc[~train_mask, 'Prediction'])


## Submission

In [ ]:

submission = pd.DataFrame({
    'PassengerId': full_df.loc[~train_mask, 'PassengerId'],
    'Survived': final_preds.astype(int)
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()
